In [1]:
import json
from pathlib import Path
import glob, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
#DEVICE = "cpu"

# your basic TCN settings
K = 25            # max number of boxes per frame
#NUM_FEATURES = 30 # number of features for every person(ex. boxes and keypoints)
NUM_FEATURES = 30
WINDOW_SIZE = 16 # ~3 seconds if ~5 jsons/sec
WINDOW_STEP = 1  # move by 1 step
NUM_CLASSES = 1#3 # 0=none, 1=tension, 2=fight
DS_PATH = '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/'

Device: cuda


In [2]:
def get_subfolders_files(path):
    """
    Returns a list of all subfolders within the given path.
    """
    p = Path(path)
    #print(p)
    subfolders = [os.path.join(path, entry.name) for entry in p.iterdir() if entry.is_dir()]
    jsons = [os.path.join(folder_path, entry)  for folder_path in subfolders for entry in os.listdir(folder_path)]
    #print(jsons)
    return jsons

#json_paths = glob.glob('/home/anna/TCN_TRAIN/DS_tension_fight_tcn/jsons_corrected/*.json')
json_paths = get_subfolders_files(DS_PATH)
print("Found jsons:", len(json_paths))

with open(json_paths[0], "r") as f:
    data_example = json.load(f)

data_example.keys()


Found jsons: 91


dict_keys(['video', 'fps', 'step', 'frames'])

In [3]:
# this depends on your format, I'm assuming something like:
# data["frames"] = [ { "detections": [...], "events": [...] }, ... ]

frame0 = data_example["frames"][0]
frame0.keys()

dict_keys(['f', 't', 'individual_events', 'group_events', 'bbs_list_of_keypoints'])

In [17]:
frame0['bbs_list_of_keypoints'][0]

[0,
 0.9067593216896057,
 0.29936739802360535,
 0.5666654109954834,
 0.40787649154663086,
 0.9314240217208862,
 [0.34645289182662964,
  0.622401773929596,
  0.3182511031627655,
  0.6586061120033264,
  0.3621777594089508,
  0.6908908486366272,
  0.0,
  0.0,
  0.37733158469200134,
  0.7644481658935547,
  0.0,
  0.0,
  0.3980977237224579,
  0.7973667979240417,
  0.32375961542129517,
  0.7825773358345032,
  0.3549858033657074,
  0.7984973192214966,
  0.3357553780078888,
  0.8383453488349915,
  0.3716656565666199,
  0.8470320701599121,
  0.3280024826526642,
  0.9030181765556335,
  0.3721572458744049,
  0.9080581665039062]]

In [5]:
def frame_to_vector(frame, K):
    """
    frame: dict for one time step
    K: max number of boxes per frame
    """
    detections = frame['bbs_list_of_keypoints']  # <-- adapt this key name

    # sort boxes by area desc
    dets_sorted = sorted(
        detections,
        key=lambda d: (d[4] - d[2]) * (d[5] - d[3]),
        reverse=True,
    )

    boxes = []
    keypoints = []
    for d in dets_sorted[:K]:
        x1, y1, x2, y2 = d[2:6]
        w = x2 - x1
        h = y2 - y1
        cx = (x1 + x2 )/ 2
        cy = (y1 + y2 )/ 2

        # normalize
        '''cx /= img_w
        cy /= img_h
        w  /= img_w
        h  /= img_h'''

        boxes.extend([cx, cy, w, h]+ d[6])

    # pad if fewer than K boxes
    if len(dets_sorted) < K:
        boxes.extend([0.0] * NUM_FEATURES * (K - len(dets_sorted)))

    n_people = len(detections)

    x_t = boxes + [float(n_people)]
    return np.array(x_t, dtype=np.float32)
    
def events_to_label(frame):
    # adjust key to your json
    events = frame.get("group_events", [])  

    if 4 in events:
        #return 2
        return 1
    if 3 in events:
        return 1
    return 0

In [6]:
def add_ramp_before_after_events(
    y,
    ramp=(0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875),
):
    """
    y: 1D numpy array of 0/1 labels (per frame) for your joined class (danger)
    ramp: values to assign to frames around each event segment.
          The value closest to the event is ramp[-1] (largest), farthest is ramp[0].
          This function does NOT change frames where orig==1 (true event frames).
    """
    y = np.asarray(y, dtype=float).copy()
    orig = (y >= 0.5).astype(np.uint8)  # safe even if y already has floats
    L = len(y)

    # event starts: index s where orig[s]=1 and previous is 0 (or s==0)
    prev = np.r_[0, orig[:-1]]
    starts = np.flatnonzero((orig == 1) & (prev == 0))

    # event ends: index e where orig[e]=1 and next is 0 (or e==L-1)
    nxt = np.r_[orig[1:], 0]
    ends = np.flatnonzero((orig == 1) & (nxt == 0))

    ramp = np.asarray(ramp, dtype=float)
    K = len(ramp)

    for j in range(1, K + 1):
        val = ramp[-j]  # closest frame gets the largest value

        # before starts
        idx = starts - j
        m = (idx >= 0)
        idx = idx[m]
        # only paint where there is no true event (prevents overwriting)
        idx = idx[orig[idx] == 0]
        y[idx] = np.maximum(y[idx], val)

        # after ends (first after is end+1)
        idx = ends + j
        m = (idx < L)
        idx = idx[m]
        idx = idx[orig[idx] == 0]  # prevents overwriting next event if too close
        y[idx] = np.maximum(y[idx], val)

    return np.clip(y, 0.0, 1.0)


In [62]:
def add_ramp_before_after_events(y, ramp=(0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0,875)):
    """
    y: 1D numpy array of 0/1 labels (per frame)
    ramp: values to assign to frames before first 1 in a segment.
          Last value in ramp (1.0) will be applied at the event frame itself
          if you want that; if not, just use (0.25, 0.5, 0.75) etc.
    """
    y = y.astype(float).copy()
    K = len(ramp)

    # find indices where an event *starts*: 0 -> 1 transition
    # event_starts: positions i where y[i] was 1 in original and (i == 0 or y[i-1] == 0)
    #orig = y.copy()  # keep original 0/1 to detect starts
    #event_starts = np.where((orig == 1) & (np.concatenate([[0], orig[:-1]]) == 0))[0]
    differences = np.diff(y)

    indices_0_to_1 = np.where(differences == 1)[0] + 1

    # Find indices where the difference is -1 (1 -> 0 transition)
    indices_1_to_0 = np.where(differences == -1)[0]
    for i in range(1, K + 1):
        y[indices_0_to_1 - i] = ramp[-i]
        y[indices_1_to_0 + i] = ramp[-i]
        
    '''for s in event_starts:
        # go backwards over ramp
        for k in range(K):
            idx = s - (K - 1 - k)  # so that ramp[-1] ends at s
            if idx < 0:
                continue
            # don't overwrite later events if there are overlapping things
            if orig[idx] == 0:  # only change non-event frames
                y[idx] = max(y[idx], ramp[k])
            else:
                # if we hit another 1 (inside the event segment), we usually stop going further back
                # but you can also `continue` if you want.
                pass'''

    return y


In [7]:
def json_to_sequence(path, K):
    with open(path, "r") as f:
        data = json.load(f)

    frames = data["frames"]  # adapt if needed

    features = []
    labels = []

    for frame in frames:
        x_t = frame_to_vector(frame, K)
        y_t = events_to_label(frame)

        features.append(x_t)
        labels.append(y_t)

    features = np.stack(features).astype(np.float32)  # (N, C)
    labels   = np.array(labels, dtype=np.float32)       # (N,)
    labels_ramped = add_ramp_before_after_events(labels)
    return features, labels_ramped


In [8]:
json_paths

['/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_16json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_19json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_10json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_21json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_20json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_27json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_12json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_30json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_5json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_15json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_1json',
 '/home/anna/TCN_TRAIN/DS_tension_fight_tcn/usual_jsons_from_events/event_23json',
 '/hom

In [9]:
print(json_paths[-2])
X_seq, y_seq = json_to_sequence(json_paths[-2], K)
X_seq.shape, y_seq.shape, y_seq

/home/anna/TCN_TRAIN/DS_tension_fight_tcn/jsons_corrected/cam7_7.json


((508, 751),
 (508,),
 array([0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.125, 0.25 , 0.375, 0.5  , 0.625, 0.75 , 0.875,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   

In [10]:
class EventJsonDataset(Dataset):
    def __init__(self, json_paths, K,
                 window_size=16, window_step=1):
        self.samples = []
        self.window_size = window_size
        self.window_step = window_step
        self.K = K
        #self.img_w = img_w
        #self.img_h = img_h

        for path in json_paths:
            X_seq, y_seq = json_to_sequence(path, K)
            N = len(X_seq)
            if N < window_size:
                continue

            for start in range(0, N - window_size + 1, window_step):
                end = start + window_size
                X_win = X_seq[start:end]    # (T, C)
                y_win = y_seq[start:end]    # (T,)
                y_central = y_win[window_size//2]
                # if window has only 'none', keep only a fraction
                #if np.all(y_win == 0):
                    #if np.random.rand() > 0.2:  # keep ~20% of pure-none windows
                        #continue

                self.samples.append((X_win, y_central))

                
        print("Total windows:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X_win, y_last = self.samples[idx]
        # convert to tensors here
        X_win = torch.from_numpy(X_win)        # (T, C)
        y_last = torch.tensor(y_last)        # (T,)
        return X_win, y_last


In [11]:
dataset = EventJsonDataset(json_paths, K,
                           window_size=WINDOW_SIZE,
                           window_step=WINDOW_STEP)
X_win, y_last = dataset[1000]
X_win.shape, y_last

Total windows: 74638


(torch.Size([16, 751]), tensor(0., dtype=torch.float64))

In [12]:
len(dataset)

74638

In [ ]:
class EventTCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_layers=3):
        super().__init__()

        layers = []
        in_ch = input_dim
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                TemporalConvBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=3,
                    dilation=dilation,
                )
            )
            in_ch = hidden_dim
            dilation *= 2

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dim, 1)   # <--- ONE logit

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)              # (B, C, T)
        feat = self.tcn(x)                 # (B, hidden_dim, T)
        last_feat = feat[:, :, -1]         # (B, hidden_dim)
        logit = self.fc(last_feat).squeeze(-1)  # (B,)  raw score
        return logit


In [13]:
class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        # x: (B, C_in, T)
        out = self.conv(x)
        out = self.bn(out)
        out = F.relu(out)
        return out

class EventTCN(nn.Module):
    def __init__(self, input_dim, num_classes=1,
                 hidden_dim=64, num_layers=3):
        super().__init__()

        layers = []
        in_ch = input_dim
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                TemporalConvBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=3,
                    dilation=dilation,
                )
            )
            in_ch = hidden_dim
            dilation *= 2  # 1,2,4,...

        self.tcn = nn.Sequential(*layers)
        #self.head = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        self.fc = nn.Linear(hidden_dim, 1)   # <--- ONE logit

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)          # (B, C, T)
        feat = self.tcn(x)             # (B, hidden_dim, T)
        #logits = self.head(feat)       # (B, num_classes, T)
        #logits = logits.transpose(1, 2)  # (B, T, num_classes)
        last_feat = feat[:, :, -1]         # (B, hidden_dim)
        logits = self.fc(last_feat).squeeze(-1)  # (B,)  raw score
        return logits


In [14]:
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = EventTCN(input_dim=NUM_FEATURES*K + 1, num_classes=NUM_CLASSES).to(DEVICE)

X_batch, y_batch = next(iter(loader))
X_batch = X_batch.to(DEVICE)  # (B, T, C)
logits = model(X_batch)       # (B, T, num_classes)
print("X_batch:", X_batch.shape)
print("logits:", logits.shape)
batches_with_y_not_0_or_1 = 0

for batch_features, batch_labels in loader:
    # Check if any label in the current batch is neither 0 nor 1
    # torch.logical_and checks element-wise for both conditions
    # .any() checks if at least one element in the resulting boolean tensor is True
    if torch.logical_and(batch_labels != 0, batch_labels >= 0).any():
        batches_with_y_not_0_or_1 += 1

print(f"Number of batches with labels not 0 or 1: {batches_with_y_not_0_or_1}")

X_batch: torch.Size([64, 16, 751])
logits: torch.Size([64])
Number of batches with labels not 0 or 1: 1167


In [15]:
from sklearn.model_selection import train_test_split

train_paths, val_paths = train_test_split(json_paths, test_size=0.2, random_state=42)

train_dataset = EventJsonDataset(train_paths, K,
                                 window_size=WINDOW_SIZE,
                                 window_step=WINDOW_STEP)

val_dataset = EventJsonDataset(val_paths, K,
                               window_size=WINDOW_SIZE,
                               window_step=WINDOW_STEP)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
batches_with_y_not_0_or_1 = 0

for batch_features, batch_labels in train_loader:
    # Check if any label in the current batch is neither 0 nor 1
    # torch.logical_and checks element-wise for both conditions
    # .any() checks if at least one element in the resulting boolean tensor is True
    if torch.logical_and(batch_labels != 0, batch_labels != 1).any():
        batches_with_y_not_0_or_1 += 1

print(f"Number of batches with labels not 0 or 1: {batches_with_y_not_0_or_1}")
batches_with_y_not_0_or_1 = 0

for batch_features, batch_labels in val_loader:
    # Check if any label in the current batch is neither 0 nor 1
    # torch.logical_and checks element-wise for both conditions
    # .any() checks if at least one element in the resulting boolean tensor is True
    if torch.logical_and(batch_labels != 0, batch_labels != 1).any():
        batches_with_y_not_0_or_1 += 1

print(f"Number of batches with labels not 0 or 1: {batches_with_y_not_0_or_1}")

Total windows: 61087
Total windows: 13551
Number of batches with labels not 0 or 1: 734
Number of batches with labels not 0 or 1: 21


In [ ]:
def compute_class_weights(dataset, num_classes=3):
    import numpy as np
    all_y = []
    for _, y_win in dataset:
        all_y.append(y_win.numpy())
    all_y = np.concatenate(all_y)

    counts = np.bincount(all_y, minlength=num_classes)
    total = counts.sum()
    weights = total / (num_classes * counts.clip(min=1))  # avoid /0
    return weights

weights_np = compute_class_weights(train_dataset, NUM_CLASSES)
class_weights = torch.tensor(weights_np, dtype=torch.float32, device=DEVICE)
print("class_weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight= torch.tensor([ 1.0, 4.0], dtype=torch.float32, device=DEVICE))#class_weights)


In [ ]:
import numpy as np

all_y = []
for _, y in train_dataset:
    all_y.append(y)  # y is 0 or 1
all_y = np.array(all_y)
n_neg = (all_y == 0).sum()
n_pos = (all_y > 0).sum()
print("n_neg, n_pos =", n_neg, n_pos)
# Create a boolean mask for elements > 0
condition_gt_0 = all_y > 0

# Create a boolean mask for elements < 1
condition_lt_1 = all_y < 1

# Combine the conditions using the logical AND operator (&)
combined_condition = condition_gt_0 & condition_lt_1
all_y[combined_condition]

In [1]:
model = EventTCN(input_dim=4*NUM_FEATURES + 1, num_classes=NUM_CLASSES).to(DEVICE)
#criterion = nn.CrossEntropyLoss()
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 3
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)   # (B, T, C)
        y_batch = y_batch.float().to(DEVICE)   # (B, )

        logits = model(X_batch)        # (B, 1)
        
        loss = criterion(logits, y_batch)
        #B, T, Cc = logits.shape

        #loss = criterion(
         #   logits.reshape(B*T, Cc),
         #   y_batch.view(-1)
        #)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * B# * T

        preds = logits#.argmax(dim=-1)          # (B, T)
        correct += (preds == y_batch).sum().item()
        total   += B# * T

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- validation ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)
            B, T, Cc = logits.shape

            loss = criterion(
                logits.reshape(B*T, Cc),
                y_batch.view(-1)
            )
            val_running_loss += loss.item() * B * T

            preds = logits.argmax(dim=-1)
            val_correct += (preds == y_batch).sum().item()
            val_total   += B * T

    val_loss = val_running_loss / val_total
    val_acc  = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}")


NameError: name 'EventTCN' is not defined

In [16]:
import copy

model = EventTCN(input_dim=NUM_FEATURES*K + 1, num_classes=NUM_CLASSES).to(DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(4.0, device=DEVICE))  # pos_weight is a scalar tensor
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
PATIENCE = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_loss = float("inf")
best_val_acc = 0.0

best_model_state = None
best_model_state_acc = None
epochs_no_improve = 0
best_epoch = 0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        # X_batch: (B, T, C)
        # y_batch: (B,)  in [0,1] (ramp) or {0,1}
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).float()

        logits = model(X_batch)          # (B,) raw scores
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        B = y_batch.size(0)
        running_loss += loss.item() * B

        # accuracy: binarize both labels and predictions
        with torch.no_grad():
            probs = torch.sigmoid(logits)                # (B,)
            preds = (probs >= 0.5).float()               # 0 or 1
            labels_bin = (y_batch >= 0.5).float()        # for ramp labels

            correct += (preds == labels_bin).sum().item()
            total   += B

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- validation ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()

            logits = model(X_batch)       # (B,)
            loss = criterion(logits, y_batch)

            B = y_batch.size(0)
            val_running_loss += loss.item() * B

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            labels_bin = (y_batch >= 0.5).float()

            val_correct += (preds == labels_bin).sum().item()
            val_total   += B

    val_loss = val_running_loss / val_total
    val_acc  = val_correct / val_total
        
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state_acc = copy.deepcopy(model.state_dict())
        
    # --- early stopping (optional) ---
    if PATIENCE is not None and epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}, "
              f"no improvement for {PATIENCE} epochs. Best model by loss saved after {best_epoch} epoch/s.")
        break

PATH = "tcn_best_by_loss.pt"  # Choose a suitable file path and name
torch.save(best_model_state, PATH)
torch.save(best_model_state_acc,  "tcn_best_by_acc.pth")

Epoch 01 | train_loss=0.5881, train_acc=0.831 | val_loss=0.7514, val_acc=0.720
Epoch 02 | train_loss=0.4674, train_acc=0.875 | val_loss=0.8411, val_acc=0.758
Epoch 03 | train_loss=0.3819, train_acc=0.904 | val_loss=0.8423, val_acc=0.837
Epoch 04 | train_loss=0.3166, train_acc=0.922 | val_loss=1.0489, val_acc=0.727
Epoch 05 | train_loss=0.2704, train_acc=0.936 | val_loss=1.0167, val_acc=0.809
Epoch 06 | train_loss=0.2297, train_acc=0.945 | val_loss=0.9188, val_acc=0.756
Epoch 07 | train_loss=0.2032, train_acc=0.953 | val_loss=1.2339, val_acc=0.819
Epoch 08 | train_loss=0.1829, train_acc=0.959 | val_loss=0.9044, val_acc=0.814
Epoch 09 | train_loss=0.1611, train_acc=0.964 | val_loss=1.2175, val_acc=0.851
Early stopping at epoch 9, no improvement for 8 epochs. Best model by loss saved after 0 epoch/s.


In [ ]:
import numpy as np

def evaluate_per_class(model, loader, num_classes=3):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()

            logits = model(X_batch)
            probs = torch.sigmoid(logits)
            #preds = logits.argmax(dim=-1)  # (B, T)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)

    for c, name in zip(range(2), ["none", "tension"]):
        mask = (y_true == c)
        if mask.sum() == 0:
            print(f"class {c} ({name}): no true examples")
            continue

        tp = ((y_true == c) & (y_pred == c)).sum()
        fn = ((y_true == c) & (y_pred != c)).sum()
        fp = ((y_true != c) & (y_pred == c)).sum()

        recall = tp / (tp + fn + 1e-8)
        precision = tp / (tp + fp + 1e-8)

        print(f"class {c} ({name}): "
              f"precision={precision:.3f}, recall={recall:.3f}, support={mask.sum()}")

evaluate_per_class(model, val_loader, num_classes=2)


In [18]:
import numpy as np
import torch

def evaluate_per_class(model_path, loader, threshold=0.5):
    """
    Binary evaluation: class 0 = none, class 1 = tension/event
    threshold: decision threshold on sigmoid(prob) for event.
    """
    # Load the state dictionary
    state_dict = torch.load(model_path, weights_only=True)

    # Load the state dictionary into the model
    model.load_state_dict(state_dict)
    model.eval()
    all_scores = []   # sigmoid outputs
    all_targets = []  # true labels (possibly soft) from dataset

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()   # (B,)

            logits = model(X_batch)                # (B,)
            probs = torch.sigmoid(logits)          # (B,)

            all_scores.append(probs.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    y_true = np.concatenate(all_targets)           # soft labels in [0,1]
    y_score = np.concatenate(all_scores)           # model scores in [0,1]

    # Binarize both: decide what is "event" vs "none"
    y_true_bin = (y_true >= threshold).astype(int)       # ground truth 0/1
    y_pred_bin = (y_score >= threshold).astype(int)  # predictions 0/1

    for c, name in zip([0, 1], ["none", "tension"]):
        mask = (y_true_bin == c)
        support = mask.sum()
        if support == 0:
            print(f"class {c} ({name}): no true examples")
            continue

        tp = ((y_true_bin == c) & (y_pred_bin == c)).sum()
        fn = ((y_true_bin == c) & (y_pred_bin != c)).sum()
        fp = ((y_true_bin != c) & (y_pred_bin == c)).sum()

        recall = tp / (tp + fn + 1e-8)
        precision = tp / (tp + fp + 1e-8)

        print(
            f"class {c} ({name}): "
            f"precision={precision:.3f}, recall={recall:.3f}, support={support}"
        )

# usage
#evaluate_per_class(model, val_loader, threshold=0.5)
for th in [0.5, 0.6, 0.7, 0.8]:
    print(f"\nThreshold = {th}")
    evaluate_per_class('tcn_best_by_acc.pth', val_loader, threshold=th)
for th in [0.5, 0.6, 0.7, 0.8]:
    print(f"\nThreshold = {th}")
    evaluate_per_class('tcn_best_by_loss.pt', val_loader, threshold=th)


Threshold = 0.5


/home/anna/.local/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


class 0 (none): precision=0.911, recall=0.908, support=11153
class 1 (tension): precision=0.577, recall=0.587, support=2398

Threshold = 0.6
class 0 (none): precision=0.906, recall=0.923, support=11168
class 1 (tension): precision=0.602, recall=0.549, support=2383

Threshold = 0.7
class 0 (none): precision=0.901, recall=0.933, support=11183
class 1 (tension): precision=0.620, recall=0.516, support=2368

Threshold = 0.8
class 0 (none): precision=0.894, recall=0.948, support=11200
class 1 (tension): precision=0.651, recall=0.465, support=2351

Threshold = 0.5
class 0 (none): precision=0.954, recall=0.694, support=11153
class 1 (tension): precision=0.372, recall=0.843, support=2398

Threshold = 0.6
class 0 (none): precision=0.928, recall=0.788, support=11168
class 1 (tension): precision=0.418, recall=0.713, support=2383

Threshold = 0.7
class 0 (none): precision=0.899, recall=0.874, support=11183
class 1 (tension): precision=0.474, recall=0.535, support=2368

Threshold = 0.8
class 0 (none

In [ ]:
X_win, y_win = val_dataset[0]    # (T, C), (T,)
X_win = X_win.unsqueeze(0).to(DEVICE)  # add batch dim: (1, T, C)
y_win = y_win.numpy()                  # for plotting

model.eval()
with torch.no_grad():
    logits = model(X_win)        # (1, T, num_classes)
    preds = logits.argmax(dim=-1).cpu().numpy().reshape(-1)  # (T,)


In [ ]:
print("True:      ", y_win)
print("Predicted: ", preds)


In [ ]:
import matplotlib.pyplot as plt

T = len(y_win)
t = range(T)

plt.figure(figsize=(10, 3))
plt.step(t, y_win, where="mid", label="true", linewidth=2)
plt.step(t, preds, where="mid", label="pred", linestyle="--")
plt.yticks([0, 1, 2], ["none", "tension", "fight"])
plt.xlabel("time step in window")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

def count_labels(dataset):
    all_y = []
    for _, y_win in dataset:      # y_win: (T,)
        all_y.append(y_win.numpy())
    all_y = np.concatenate(all_y)
    unique, counts = np.unique(all_y, return_counts=True)
    print(dict(zip(unique, counts)))

count_labels(train_dataset)
count_labels(val_dataset)
